In [1]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

model_name = "BAAI/bge-m3"
persist_dir = "../../data/vector_db"
collection_name = "test"   


embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cuda"} 
)


vector_db = Chroma(
    persist_directory=persist_dir,
    embedding_function=embedding,
    collection_name=collection_name
)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

C:\Users\huynh\AppData\Local\Temp\ipykernel_32076\669495407.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
d:\VSCode\AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\huynh\AppData\Local\Temp\ipykernel_32076\669495407.py:15: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip i

In [2]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "false"

In [ ]:

query = "Công thức hàm đối ngẫu Lagrange là gì"

docs = retriever.invoke(query)

for doc in docs:
    print(f"Document: {doc.page_content}")

Document: NGỮ CẢNH: test > Đối ngẫu > 25.2. Hàm đối ngẫu Lagrange > 25.2.2. Hàm đối ngẫu Lagrange
------------------------------
25.2.2. Hàm đối ngẫu Lagrange  
Hàm đối ngẫu Lagrange (the Lagrange dual function) của bài toán tối ưu (viết gọn là hàm số đối ngẫu) [\(25.5\)](#page-1-0) là một hàm của các biến đối ngẫu λ và ν, được định nghĩa là infimum theo x của hàm Lagrange:  
$$g(\boldsymbol{\lambda}, \boldsymbol{\nu}) = \inf_{\mathbf{x} \in \mathcal{D}} \mathcal{L}(\mathbf{x}, \boldsymbol{\lambda}, \boldsymbol{\nu}) = \inf_{\mathbf{x} \in \mathcal{D}} \left( f_0(\mathbf{x}) + \sum_{i=1}^m \lambda_i f_i(\mathbf{x}) + \sum_{j=1}^p \nu_j h_j(\mathbf{x}) \right)$$
(25.6)  
Nếu hàm Lagrange không bị chặn dưới, hàm đối ngẫu tại λ, ν lấy giá trị −∞.  
Lưu ý :  
- inf được lấy trên miền x ∈ D, tức tập xác định của bài toán. Tập xác định này khác với tập khả thi – là tập hợp các điểm thoả mãn các ràng buộc.
- Với mỗi x, hàm số đối ngẫu là một hàm affine của (λ, ν), tức là một hàm vừa lồi, vừa 

In [4]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.to('cuda') # or 'cpu' if no GPU is available
model.eval()


# Rerank documents
rerank_docs = model.rerank(
    query,
    [doc.page_content for doc in docs])

# Results are sorted by relevance score (highest first)
for result in rerank_docs:
    print(f"Score: {result['relevance_score']:.4f}")
    print(f"Document: {result['document'][:100]}...")
    print()


Score: 0.5493
Document: NGỮ CẢNH: test > Đối ngẫu > 25.2. Hàm đối ngẫu Lagrange > 25.2.2. Hàm đối ngẫu Lagrange
------------...

Score: 0.1525
Document: NGỮ CẢNH: test > Đối ngẫu > 25.2. Hàm đối ngẫu Lagrange > 25.2.1. Hàm Lagrange của bài toán tối ưu
-...

Score: 0.0653
Document: NGỮ CẢNH: test > Đối ngẫu > 25.2. Hàm đối ngẫu Lagrange > 25.2.4. Ví dụ
----------------------------...

Score: -0.0272
Document: NGỮ CẢNH: test > Đối ngẫu > 25.2. Hàm đối ngẫu Lagrange > 25.2.3. Chặn dưới của giá trị tối ưu
-----...

Score: -0.0286
Document: NGỮ CẢNH: test > Đối ngẫu > 25.1. Giới thiệu
------------------------------
25.1. Giới thiệu  
Trong...

